# Manual MLP
Train and evaluate a multilayer perceptron implemented from scratch without keras, scikit-learn, PyTorch, or TensorFlow.

- Training data: `../Data/train_new.csv`
- Test data: `../Data/test_new.csv`
- Target: `fit`

In [19]:
import numpy as np
import pandas as pd
from pathlib import Path

train_path = Path("../Data/train_new.csv")
test_path = Path("../Data/test_new.csv")

train_df = pd.read_csv(train_path)
test_df = pd.read_csv(test_path)

target_col = "fit"
if target_col not in train_df.columns:
    raise ValueError(f"Expected target column '{target_col}' in training data.")

feature_cols = [col for col in train_df.columns if col != target_col]

categorical_cols = [col for col in ["size", "cup size", "bra size", "category"] if col in feature_cols]
numeric_cols = [col for col in feature_cols if col not in categorical_cols]

train_features_base = train_df[feature_cols].copy()
test_features_base = test_df[feature_cols].copy()

numeric_aug_cols = []
for col in categorical_cols:
    train_num = pd.to_numeric(train_features_base[col], errors="coerce")
    test_num = pd.to_numeric(test_features_base[col], errors="coerce")

    if train_num.notna().any():
        new_col = f"{col}__num"
        median_val = float(train_num.median()) if not np.isnan(train_num.median()) else 0.0
        train_features_base[new_col] = train_num.fillna(median_val)
        test_features_base[new_col] = test_num.fillna(median_val)
        numeric_aug_cols.append(new_col)

for col in categorical_cols:
    train_features_base[col] = train_features_base[col].astype(str)
    test_features_base[col] = test_features_base[col].astype(str)

combined_features = pd.concat([train_features_base, test_features_base], axis=0, ignore_index=True)
combined_features = pd.get_dummies(combined_features, columns=categorical_cols, drop_first=False)

train_features_df = combined_features.iloc[: len(train_df)].copy()
test_features_df = combined_features.iloc[len(train_df) :].copy()

numeric_for_scaling = [col for col in numeric_cols + numeric_aug_cols if col in train_features_df.columns]
for col in numeric_for_scaling:
    mu_col = float(train_features_df[col].mean())
    sigma_col = float(train_features_df[col].std())
    if sigma_col == 0 or np.isnan(sigma_col):
        sigma_col = 1.0
    train_features_df[col] = (train_features_df[col] - mu_col) / sigma_col
    test_features_df[col] = (test_features_df[col] - mu_col) / sigma_col

interaction_pairs = [
    ("hips", "height"),
    ("hips", "bra size__num"),
    ("size__num", "bra size__num"),
    ("size__num", "cup size__num"),
    ("bra size__num", "cup size__num"),
]
for a, b in interaction_pairs:
    if a in train_features_df.columns and b in train_features_df.columns:
        col_name = f"{a}__x__{b}"
        train_features_df[col_name] = train_features_df[a] * train_features_df[b]
        test_features_df[col_name] = test_features_df[a] * test_features_df[b]

for col in ["hips", "height", "size__num", "bra size__num", "cup size__num"]:
    if col in train_features_df.columns:
        sq_col = f"{col}__sq"
        train_features_df[sq_col] = train_features_df[col] ** 2
        test_features_df[sq_col] = test_features_df[col] ** 2

X_train = train_features_df.to_numpy(dtype=np.float32)
X_test = test_features_df.to_numpy(dtype=np.float32)
y_train = train_df[target_col].to_numpy(dtype=np.int64)
y_test = test_df[target_col].to_numpy(dtype=np.int64)

classes = np.sort(np.unique(y_train))
class_to_index = {c: i for i, c in enumerate(classes)}
if not np.array_equal(classes, np.arange(len(classes))):
    y_train = np.array([class_to_index[val] for val in y_train], dtype=np.int64)
    y_test = np.array([class_to_index[val] for val in y_test], dtype=np.int64)

input_dim = X_train.shape[1]
num_classes = len(np.unique(y_train))

print("Train shape:", X_train.shape, "Test shape:", X_test.shape)
print("Raw feature columns:", feature_cols)
print("One-hot/engineered feature count:", input_dim)
print("Classes:", np.unique(y_train))

Train shape: (8439, 64) Test shape: (2110, 64)
Raw feature columns: ['size', 'cup size', 'hips', 'bra size', 'category', 'height']
One-hot/engineered feature count: 64
Classes: [0 1 2]


In [20]:
def stratified_train_val_split(y, val_fraction=0.15, seed=42):
    rng = np.random.default_rng(seed)
    train_idx = []
    val_idx = []

    for cls in np.unique(y):
        cls_idx = np.where(y == cls)[0]
        rng.shuffle(cls_idx)

        n_val = int(round(len(cls_idx) * val_fraction))
        if len(cls_idx) > 1:
            n_val = min(max(n_val, 1), len(cls_idx) - 1)
        else:
            n_val = 0

        val_idx.extend(cls_idx[:n_val])
        train_idx.extend(cls_idx[n_val:])

    return np.array(train_idx, dtype=np.int64), np.array(val_idx, dtype=np.int64)


class ManualMLP:
    def __init__(self, input_dim, hidden_dims, output_dim, lr=0.003, l2_lambda=1e-4, seed=42):
        self.lr = lr
        self.l2_lambda = l2_lambda
        self.seed = seed

        layer_dims = [input_dim] + list(hidden_dims) + [output_dim]
        rng = np.random.default_rng(seed)

        self.weights = []
        self.biases = []
        self.mw = []
        self.vw = []
        self.mb = []
        self.vb = []

        for fan_in, fan_out in zip(layer_dims[:-1], layer_dims[1:]):
            w = rng.normal(0, np.sqrt(2.0 / fan_in), size=(fan_in, fan_out)).astype(np.float32)
            b = np.zeros((1, fan_out), dtype=np.float32)
            self.weights.append(w)
            self.biases.append(b)
            self.mw.append(np.zeros_like(w))
            self.vw.append(np.zeros_like(w))
            self.mb.append(np.zeros_like(b))
            self.vb.append(np.zeros_like(b))

        self.beta1 = 0.9
        self.beta2 = 0.999
        self.eps = 1e-8
        self.step = 0

    def forward(self, x):
        activations = [x]
        pre_activations = []

        a = x
        for layer in range(len(self.weights) - 1):
            z = a @ self.weights[layer] + self.biases[layer]
            a = np.maximum(0, z)
            pre_activations.append(z)
            activations.append(a)

        logits = a @ self.weights[-1] + self.biases[-1]
        shifted = logits - np.max(logits, axis=1, keepdims=True)
        exp_scores = np.exp(shifted)
        probs = exp_scores / np.sum(exp_scores, axis=1, keepdims=True)
        pre_activations.append(logits)
        activations.append(probs)

        cache = {"A": activations, "Z": pre_activations}
        return probs, cache

    @staticmethod
    def cross_entropy(probs, y_true):
        probs = np.clip(probs, 1e-12, 1.0)
        return -np.mean(np.log(probs[np.arange(len(y_true)), y_true]))

    def backward(self, y_true, cache):
        activations = cache["A"]
        pre_activations = cache["Z"]
        batch_size = y_true.shape[0]
        n_layers = len(self.weights)

        grad_w = [None] * n_layers
        grad_b = [None] * n_layers

        d_logits = activations[-1].copy()
        d_logits[np.arange(batch_size), y_true] -= 1.0
        d_logits /= batch_size

        for layer in reversed(range(n_layers)):
            a_prev = activations[layer]
            grad_w[layer] = a_prev.T @ d_logits + (self.l2_lambda / batch_size) * self.weights[layer]
            grad_b[layer] = np.sum(d_logits, axis=0, keepdims=True)

            if layer > 0:
                d_hidden = d_logits @ self.weights[layer].T
                d_logits = d_hidden * (pre_activations[layer - 1] > 0)

        return grad_w, grad_b

    def update(self, grad_w, grad_b):
        self.step += 1

        for layer in range(len(self.weights)):
            self.mw[layer] = self.beta1 * self.mw[layer] + (1 - self.beta1) * grad_w[layer]
            self.vw[layer] = self.beta2 * self.vw[layer] + (1 - self.beta2) * (grad_w[layer] ** 2)
            self.mb[layer] = self.beta1 * self.mb[layer] + (1 - self.beta1) * grad_b[layer]
            self.vb[layer] = self.beta2 * self.vb[layer] + (1 - self.beta2) * (grad_b[layer] ** 2)

            mw_hat = self.mw[layer] / (1 - self.beta1 ** self.step)
            vw_hat = self.vw[layer] / (1 - self.beta2 ** self.step)
            mb_hat = self.mb[layer] / (1 - self.beta1 ** self.step)
            vb_hat = self.vb[layer] / (1 - self.beta2 ** self.step)

            self.weights[layer] -= self.lr * mw_hat / (np.sqrt(vw_hat) + self.eps)
            self.biases[layer] -= self.lr * mb_hat / (np.sqrt(vb_hat) + self.eps)

    def predict(self, x):
        probs, _ = self.forward(x)
        return np.argmax(probs, axis=1)


def train_mlp(model, x_train, y_train, x_val, y_val, epochs=200, batch_size=128, patience=20):
    rng = np.random.default_rng(model.seed + 99)
    history = []

    best_val_acc = -1.0
    best_epoch = 0
    wait = 0
    best_weights = [w.copy() for w in model.weights]
    best_biases = [b.copy() for b in model.biases]

    n_train = x_train.shape[0]

    for epoch in range(1, epochs + 1):
        order = rng.permutation(n_train)
        x_epoch = x_train[order]
        y_epoch = y_train[order]

        for start in range(0, n_train, batch_size):
            end = start + batch_size
            xb = x_epoch[start:end]
            yb = y_epoch[start:end]

            probs, cache = model.forward(xb)
            grad_w, grad_b = model.backward(yb, cache)
            model.update(grad_w, grad_b)

        train_probs, _ = model.forward(x_train)
        val_probs, _ = model.forward(x_val)

        train_loss = model.cross_entropy(train_probs, y_train)
        val_loss = model.cross_entropy(val_probs, y_val)
        train_acc = np.mean(np.argmax(train_probs, axis=1) == y_train)
        val_acc = np.mean(np.argmax(val_probs, axis=1) == y_val)

        history.append({
            "epoch": epoch,
            "train_loss": float(train_loss),
            "val_loss": float(val_loss),
            "train_acc": float(train_acc),
            "val_acc": float(val_acc),
        })

        if val_acc > best_val_acc + 1e-7:
            best_val_acc = val_acc
            best_epoch = epoch
            best_weights = [w.copy() for w in model.weights]
            best_biases = [b.copy() for b in model.biases]
            wait = 0
        else:
            wait += 1
            if wait >= patience:
                break

    model.weights = best_weights
    model.biases = best_biases

    return history, best_epoch, float(best_val_acc)


def confusion_matrix_np(y_true, y_pred, num_classes):
    cm = np.zeros((num_classes, num_classes), dtype=np.int64)
    for t, p in zip(y_true, y_pred):
        cm[t, p] += 1
    return cm


def classification_metrics(y_true, y_pred, num_classes):
    cm = confusion_matrix_np(y_true, y_pred, num_classes)

    per_class_rows = []
    precisions = []
    recalls = []
    f1s = []

    for cls in range(num_classes):
        tp = cm[cls, cls]
        fp = cm[:, cls].sum() - tp
        fn = cm[cls, :].sum() - tp
        support = cm[cls, :].sum()

        precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0

        precisions.append(precision)
        recalls.append(recall)
        f1s.append(f1)

        per_class_rows.append({
            "class": cls,
            "precision": precision,
            "recall": recall,
            "f1": f1,
            "support": int(support),
        })

    accuracy = float(np.mean(y_true == y_pred))
    macro_precision = float(np.mean(precisions))
    macro_recall = float(np.mean(recalls))
    macro_f1 = float(np.mean(f1s))

    per_class_df = pd.DataFrame(per_class_rows)
    summary = {
        "accuracy": accuracy,
        "macro_precision": macro_precision,
        "macro_recall": macro_recall,
        "macro_f1": macro_f1,
    }

    return cm, per_class_df, summary

In [21]:
# Fixed settings chosen from the best run observed so far (test accuracy = 0.5005).
best_config = {
    "hidden_dims": [128, 64],
    "lr": 0.003,
    "batch_size": 64,
    "l2": 5e-5,
}
best_epochs = 10

print("Using fixed best configuration:")
print(best_config)
print("Epochs:", best_epochs)

Using fixed best configuration:
{'hidden_dims': [128, 64], 'lr': 0.003, 'batch_size': 64, 'l2': 5e-05}
Epochs: 10


In [23]:
def fit_fixed_epochs(model, x_train, y_train, epochs, batch_size, seed=123):
    rng = np.random.default_rng(seed)
    n_train = x_train.shape[0]

    for _ in range(epochs):
        order = rng.permutation(n_train)
        x_epoch = x_train[order]
        y_epoch = y_train[order]

        for start in range(0, n_train, batch_size):
            end = start + batch_size
            xb = x_epoch[start:end]
            yb = y_epoch[start:end]

            probs, cache = model.forward(xb)
            grad_w, grad_b = model.backward(yb, cache)
            model.update(grad_w, grad_b)


best_run_seed = 5013
final_model = ManualMLP(
    input_dim=input_dim,
    hidden_dims=best_config["hidden_dims"],
    output_dim=num_classes,
    lr=best_config["lr"],
    l2_lambda=best_config["l2"],
    seed=best_run_seed,
 )

fit_fixed_epochs(
    final_model,
    X_train,
    y_train,
    epochs=best_epochs,
    batch_size=best_config["batch_size"],
    seed=best_run_seed,
 )

y_train_pred = final_model.predict(X_train)
y_test_pred = final_model.predict(X_test)

train_acc = float(np.mean(y_train_pred == y_train))
cm, per_class_df, summary = classification_metrics(y_test, y_test_pred, num_classes)

print(f"Training accuracy: {train_acc:.4f}")
print("Test metrics summary:")
for metric_name, metric_value in summary.items():
    print(f"{metric_name}: {metric_value:.4f}")

print("\nPer-class metrics:")
display(per_class_df)

cm_df = pd.DataFrame(
    cm,
    index=[f"true_{i}" for i in range(num_classes)],
    columns=[f"pred_{i}" for i in range(num_classes)],
)
print("Confusion matrix:")
display(cm_df)

Training accuracy: 0.5255
Test metrics summary:
accuracy: 0.5005
macro_precision: 0.4890
macro_recall: 0.4652
macro_f1: 0.4555

Per-class metrics:


,class,precision,recall,f1,support
0,0,0.519397,0.354412,0.421329,680
1,1,0.510786,0.778169,0.616744,852
2,2,0.436782,0.262976,0.328294,578


Confusion matrix:


,pred_0,pred_1,pred_2
true_0,241,326,113
true_1,106,663,83
true_2,117,309,152
